In [2]:
import chipwhisperer as cw
import time
import numpy as np
import struct


scope = cw.scope()
bitfile1 = "/home/40265864@ecit.qub.ac.uk/FFT_Toeplitz/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
bitfile2 = "/home/40265864@ecit.qub.ac.uk/FFT_Toeplitz/cw305_fft/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
bitfile3 = "/home/40265864@ecit.qub.ac.uk/FFT_Toeplitz/fft_16_bit/cw305/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
target = cw.target(scope, cw.targets.CW305, bsfile=bitfile3, force=True, fpga_id='100t', platform='cw305')
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.freq_ctr                     changed from 10200920                  to 15000514                 
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_error                 changed from True                      to False                    
scope.clock.extclk_

In [3]:
scope.adc.stream_mode = "segmented"
scope.adc.samples = 256
scope.adc.bits_per_sample = 12
scope.adc.segments = 12
target.pll.pll_outfreq_set(15E6, 1)
target._clksleeptime = 100
scope.gain.db = 45
scope.adc.timeout = 10

scope.clock.clkgen_freq = 15e6
scope.clock.clkgen_src = 'extclk'
scope.clock.adc_mul = 4

scope.adc.offset = 3

target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)
target.pll.pll_outfreq_set(15E6, 1)  # FPGA clock

# ensure ADC is locked:
scope.clock.reset_adc()
assert (scope.clock.adc_locked), "ADC failed to lock"

In [ ]:
#"""
scope.adc.samples = 500
#scope.adc.segments = 12
scope.adc.bits_per_sample = 8
#scope.adc.segment_cycles = 3
scope.adc.offset = 0
scope.adc.stream_mode = True#"segmented"  
#scope.adc.segment_cycle_counter_en = True
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
#scope.trigger.module = "basic"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"
scope.gain.mode = "high"
target._clksleeptime = 100
scope.gain.gain = 45
scope.adc.timeout = 5  # in milliseconds
#"""

In [ ]:
#"""
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(10E6, 1)

# 1ms is plenty of idling time
target.clkusbautooff = True
target.clksleeptime = 10

scope.clock.clkgen_freq = 40e6
scope.clock.adc_src = 'extclk_x4'
scope.clock.adc_mul = 2
scope.clock.reset_adc()
#"""

In [4]:
ktp = cw.ktp.Basic()
key, text = ktp.next()
#key = b'\x06\x55\x37'
reg_crypt_textin = reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "01","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","01","00"])




reg_crypt_key1 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key2 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "01","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key3 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","01","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key4 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "01","00","01","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])



reg_reset =  ["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"]


data_bytes_text_in = list([int(byte,16) for byte in reg_crypt_textin])
data_bytes_key_in1 = list([int(byte,16) for byte in reg_crypt_key1])
data_bytes_key_in2 = list([int(byte,16) for byte in reg_crypt_key2])
data_bytes_key_in3 = list([int(byte,16) for byte in reg_crypt_key3])
data_bytes_key_in4 = list([int(byte,16) for byte in reg_crypt_key4])

data_bytes_reset = list([int(byte,16) for byte in reg_reset])

In [6]:
scope.arm()
target.fpga_write(target.REG_CRYPT_KEY,data_bytes_text_in)
target.fpga_write(target.REG_CRYPT_TEXTIN,data_bytes_text_in)
target.go()
ret = scope.capture()
trace = scope.get_last_trace()


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:695) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid


In [5]:
res = target.readOutput()
print(res.hex())

0000000000000000000000000000000000000000000000000000000000000000


In [ ]:
oplen = scope.adc.trig_count
print('Operation length: %d cycles' % oplen)

In [ ]:
cw.plot(trace)

In [ ]:
def my_capture(key):
    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY,key)
    target.fpga_write(target.REG_CRYPT_TEXTIN,key)
    target.go()
    oplen = scope.adc.trig_count
    print('Operation length: %d cycles' % oplen)
    ret = scope.capture()
    trace = scope.get_last_trace()
    return trace

In [ ]:
def align_traces(traces):
    ref_trace = traces[0]  # Use the first trace as reference
    aligned_traces = []

    for trace in traces:
        correlation = np.correlate(trace, ref_trace, mode="valid")  # Compute cross-correlation
        shift = np.argmax(correlation) - (len(trace) - 1)  # Find best alignment
        aligned_trace = np.roll(trace, -shift)  # Shift the trace
        aligned_traces.append(aligned_trace)
    
    return np.array(aligned_traces)

In [ ]:
traces1 = []
traces2 = []
traces3 = []
traces4 = []

for _ in range(10):
    traces1.append(my_capture(data_bytes_key_in1))
    traces2.append(my_capture(data_bytes_key_in2))
    traces3.append(my_capture(data_bytes_key_in3))
    traces4.append(my_capture(data_bytes_key_in4))



In [ ]:
traces1 = align_traces(traces1)
traces2 = align_traces(traces2)
traces3 = align_traces(traces3)
traces4 = align_traces(traces4)


trace1_avg = np.mean(traces1, axis=0)
trace2_avg = np.mean(traces2, axis=0)
trace3_avg = np.mean(traces3, axis=0)
trace4_avg = np.mean(traces4, axis=0)

In [ ]:
cw.plot(traces1[1]) * cw.plot(traces1[5])

In [ ]:
cw.plot(trace1_avg) * cw.plot(trace2_avg) * cw.plot(trace3_avg) * cw.plot(trace4_avg)

In [ ]:
cw.plot(trace1_avg - traces1[0]) * cw.plot(traces1[3] - traces2[3])

In [ ]:
print(f"Captured {scope.adc.trig_count} segments")


In [ ]:
cw.plot(trace0 - trace1)

In [ ]:
fig = cw.plot()
for t in trace:
    fig *= cw.plot(t)
fig

In [ ]:
#res = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)
res = target.readOutput()
print(res.hex())

In [ ]:
def get_traces(key, text):

    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY, key)
    target.fpga_write(target.REG_CRYPT_TEXTIN, text)    
    target.usb_trigger_toggle()
    target.go()
    ret = scope.capture()
    trace = scope.get_last_trace_segmented()
    return trace

In [ ]:
trace = get_traces(key, text)

In [ ]:
cw.plot(trace)

In [ ]:
#target.fpga_write(target.REG_CRYPT_KEY,key)
#target.fpga_write(target.REG_CRYPT_TEXTIN,text)


key_on_board = target.fpga_read(target.REG_CRYPT_KEY,32)
text_on_board = target.fpga_read(target.REG_CRYPT_TEXTIN,32)
output_on_board = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)


print("Before Set")
print(text_on_board.hex())
print(key_on_board.hex())
print(output_on_board.hex())



target.fpga_write(target.REG_CRYPT_TEXTIN,data_bytes_text_in)
target.fpga_write(target.REG_CRYPT_KEY,data_bytes_key_in)

scope.arm()

target.fpga_write(target.REG_CRYPT_GO,"\x01")
target.fpga_write(target.REG_CRYPT_GO,"\x00")

ret = scope.capture()
time.sleep(0.5)

key_on_board = target.fpga_read(target.REG_CRYPT_KEY,32)
text_on_board = target.fpga_read(target.REG_CRYPT_TEXTIN,32)
output_on_board = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)

print("\nAfter Set")
print(text_on_board.hex())
print(key_on_board.hex())
print(output_on_board.hex())


In [ ]:
def get_traces(key):

    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY, key)
    target.fpga_write(target.REG_USER_LED, [0x01])    
    #target.usb_trigger_toggle()
    target.go()
    ret = scope.capture()
    trace = scope.get_last_trace()
    target.fpga_write(target.REG_USER_LED, [0x00])    
    return trace



In [ ]:
target.fpga_write(target.REG_CRYPT_KEY, data_bytes_reset)
target.go()
res = target.readOutput()
print(res.hex())

In [ ]:
traces = get_traces(data_bytes_key_in)


In [ ]:
res = target.readOutput()
print(res.hex())

In [ ]:
cw.plot(traces)